# 📦 Eniac Rabattstrategie — Gesamtnotebook (durchgehender Lauf, konsistente Namenskonvention)

**Ein Durchlauf, drei Phasen, klare Namensgrenzen:**
- Teil A (Bereinigung): `orders_cl`, `orderlines_cl`, `products_cl`, `brands_cl`
- Teil B (Qualität): `orders_qu`, `orderlines_qu`, `products_qu`, `brands_qu`
- Teil C (Analyse + Kategorien): `orders_analyse`, `orderlines_analyse`, `products_analyse`, `brands_analyse`, gemergt: `orderlines_products_analyse`

Originaldaten werden nur EINMAL in Teil A geladen. Jede Phase erstellt am Anfang eine **eigene `.copy()`** von den Ergebnissen der Vorphase (🔗 Brücken-Zellen) — nie werden Originale oder Vorphasen-Daten direkt weiterbearbeitet.

Teil D (Visualisierung/Speichern als Bilddatei) folgt **erst, wenn final entschieden ist**, was analysiert und präsentiert wird — bewusst noch nicht Teil dieses Notebooks.

⚠️ Export-Zeilen (`.to_csv`, `files.download`) sind auskommentiert — bei Bedarf gezielt aktivieren.

---
# TEIL A — Bereinigung (Meilenstein 1–3) → Ergebnis: *_cl

# 01 – Eniac Rabattstrategie: Datenbereinigung
**Meilenstein 1–3** (siehe Projektplan)

Ziel: Rohdaten einlesen, explorieren und grundlegend bereinigen (Duplikate, Datentypen, fehlende Werte).

**Inhaltsverzeichnis**
1. Daten einlesen (Original)
2. Copy erstellen
3. Exploration der Daten
4. Duplikate
5. Datentypen prüfen
6. Fehlende Werte prüfen
7. Speichern

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_rows', 1000)

## 1. Daten einlesen (Original) — Meilenstein 1

### 1.1 orderlines.csv

In [2]:
url = "https://drive.google.com/file/d/1FXjNA6uoI5-Vw94FXxDLHDa9mNIXSt1M/view?usp=sharing" # orderlines.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
df_orderlines = pd.read_csv(path)

### 1.2 orders.csv

In [3]:
url = "https://drive.google.com/file/d/1eTW_Cyo1GN8DZVLxgveiOF2uDoy8OC81/view?usp=sharing" # orders.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
df_orders = pd.read_csv(path)

### 1.3 brands.csv

In [4]:
url = "https://drive.google.com/file/d/1NP2U4cLvWYRCatSn6MJxmRms4UDlmFPd/view?usp=drive_link" # brands.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
df_brands = pd.read_csv(path)

### 1.4 products.csv

In [5]:
url = "https://drive.google.com/file/d/16l_HVmRcbV5n7mWk45xKZ74KvUuvPGlW/view?usp=sharing" # products.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
df_products = pd.read_csv(path)

## 2. Copy erstellen

In [6]:
orderlines_cl = df_orderlines.copy()
orders_cl = df_orders.copy()
brands_cl = df_brands.copy()
products_cl = df_products.copy()

## 3. Exploration der Daten

In [7]:
orderlines_cl.head()

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38


In [8]:
orders_cl.head(5)

,order_id,created_date,total_paid,state
0,241319,2017-01-02 13:35:40,44.99,Cancelled
1,241423,2017-11-06 13:10:02,136.15,Completed
2,242832,2017-12-31 17:40:03,15.76,Completed
3,243330,2017-02-16 10:59:38,84.98,Completed
4,243784,2017-11-24 13:35:19,157.86,Cancelled


In [9]:
brands_cl.head(5)

,short,long
0,8MO,8Mobility
1,ACM,Acme
2,ADN,Adonit
3,AII,Aiino
4,AKI,Akitio


In [10]:
products_cl.head(5)

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364


**Ergebnis:**
- `desc` hat 7 fehlende Werte
- `price` hat 46 fehlende Werte
- `type` ist ein String (Kategorie-/Produkttyp-Code, keine Rechenoperation nötig), 50 fehlende Werte
- `in_stock` ist der Lagerstatus

### 3.1 `.info()`, `.unique()`

In [11]:
orderlines_cl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


In [12]:
orders_cl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


In [13]:
brands_cl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 187 entries, 0 to 186
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   short   187 non-null    object
 1   long    187 non-null    object
dtypes: object(2)
memory usage: 3.1+ KB


In [14]:
products_cl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          19326 non-null  object
 1   name         19326 non-null  object
 2   desc         19319 non-null  object
 3   price        19280 non-null  object
 4   promo_price  19326 non-null  object
 5   in_stock     19326 non-null  int64 
 6   type         19276 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.0+ MB


In [15]:
products_cl['sku'].unique()

array(['RAI0007', 'APP0023', 'APP0025', ..., 'THU0061', 'THU0062',
       'THU0063'], dtype=object)

### 3.2 Ungewöhnliche Werte prüfen
Beispiel: `APP2302` hat `type` = `1,02E+12` — das ist wissenschaftliche Notation (1,02 × 10¹²), typisch wenn Excel eine lange Zahl automatisch formatiert hat. **Fazit:** separat betrachten, nicht direkt korrigierbar in dieser Phase.

### 3.3 Preis-Spalte (`products`) explorativ prüfen

In [16]:
products_cl['price'].str.count(r"\.").value_counts()

,count
price,
0.00,11528
1.00,7321
2.00,431


In [17]:
mult_decimal_rows = (products_cl['price'].str.count(r"\.") > 1).sum()
percent_corrupted = (100 * mult_decimal_rows / products_cl.shape[0])
print(f"{percent_corrupted:.2f}% der Zeilen haben mehrere Dezimalstellen im price")

2.23% der Zeilen haben mehrere Dezimalstellen im price


**Ergebnis:**
- 0 Punkte: wahrscheinlich Ganzzahl-Preise ohne Dezimalstelle (z. B. "59")
- 1 Punkt: normale Preise (z. B. "59.99")
- 2 Punkte: korrupte Preise (z. B. "26.155.941")

In [18]:
mask_corrupted = products_cl['price'].str.count(r"\.") > 1
products_cl.loc[mask_corrupted, 'price']

,price
665,1.639.792
792,4.694.994
797,4.090.042
827,2.199.791
885,5.609.698
898,69.989.909
941,69.989.909
943,2.099.895
1057,1.329.911
1058,1.599.862


**Ergebnis:**
- Der letzte Punkt ist vermutlich das Dezimalzeichen, die anderen sind Tausendertrennzeichen (z. B. "1.639.792" → 1639,79 €).
- **Entscheidung:** Korrektur erfolgt NICHT hier, sondern in Meilenstein 4 (Qualität) — siehe Projektplan "Nachträglich bereinigt".

## 4. Duplikate

### 4.1 Duplikate anschauen

In [19]:
mask_duplicated = products_cl.duplicated()
products_cl.loc[mask_duplicated]

,sku,name,desc,price,promo_price,in_stock,type
101,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
102,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
103,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
104,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
105,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
...,...,...,...,...,...,...,...
16831,APP2302,"Apple MacBook Pro 13 ""Core i5 Touch Bar 33GHz ...",New MacBook Pro 13-inch Core i5 Touch Bar 33 G...,26.155.941,26.155.941,0,"1,02E+12"
16833,APP2303,"Apple MacBook Pro 13 ""Core i5 Touch Bar 33GHz ...",New MacBook Pro 13 inch Touch Bar 33 GHz Core ...,237.559.421,23.755.942,0,"1,02E+12"
18190,PAR0077,Parrot Bebop Drone 2 Power,Drone cuadricóptero quality camera integrated ...,699.9,6.733.892,0,11905404
18308,NKI0010,Nokia Wireless sphygmomanometer Plata,Sphygmomanometer for iPhone iPad and iPod App.,129.99,1.149.899,1,11905404


### 4.2 Duplikate prüfen

In [20]:
orderlines_cl.duplicated().sum()

np.int64(0)

In [21]:
orders_cl.duplicated().sum()

np.int64(0)

In [22]:
products_cl.duplicated().sum()

np.int64(8746)

In [23]:
num_duplicated = products_cl.duplicated().sum()
total_rows = products_cl.shape[0]
percent_duplicated = (100 * num_duplicated / total_rows)
print(f"{percent_duplicated:.2f}% der Zeilen sind Duplikate")

45.26% der Zeilen sind Duplikate


In [24]:
products_cl.duplicated().value_counts(normalize=True)

,proportion
False,0.55
True,0.45


**Ergebnis:** In `orders` und `orderlines` gibt es keine Duplikate. In `products` sind über 45 % der Zeilen Duplikate.

### 4.3 Duplikate entfernen

In [25]:
products_cl = products_cl.drop_duplicates()
products_cl.shape

(10580, 7)

**Ergebnis:** Nach Entfernung 10.580 Zeilen in `products`, davon 377 (3,56 %) mit korrupten Mehrfach-Punkt-Preisen (Korrektur folgt in Meilenstein 4).

## 5. Datentypen prüfen

### 5.1 Bestellungen — `created_date`

In [26]:
orders_cl["created_date"] = pd.to_datetime(orders_cl["created_date"])

**Ergebnis:** Datentyp jetzt Datum/Uhrzeit — nach dem Import kontrollieren.

### 5.2 Auftragszeilen — `date`

In [27]:
orderlines_cl["date"] = pd.to_datetime(orderlines_cl["date"])

### 5.3 Auftragszeilen — `unit_price` (float)


Im Datensatz wurden fehlerhafte Preisformate gefunden (z. B. `34.56.546`), die nicht direkt in numerische Werte umgewandelt werden konnten.

Anstatt diese Datensätze zu löschen, entschied sich unser Team für eine Korrektur der Preisangaben. Der Grund dafür war, dass diese Produkte mit **36169 Bestellpositionen** verknüpft sind, darunter **3561 abgeschlossene Bestellungen (Completed)**.

Daher wurde die erste fehlerhafte Trennstelle entfernt, die Preise auf zwei Dezimalstellen gerundet und anschließend in numerische Werte umgewandelt.

### 1- Fehlerhafte unit_Preise identifizieren

Betroffene Bestellungen analysieren.
Zunächst werden alle Produkte identifiziert, deren Preis mehr als einen Punkt enthält.

In [28]:
unit_price_mask = orderlines_cl['unit_price'].str.count(r'\.') > 1
umsätze_zwei_punkte = orderlines_cl.loc[unit_price_mask,:]
orderlines_cl.loc[unit_price_mask, 'unit_price'].shape[0]

36169

### 2-⁠ ⁠Auswirkungen auf abgeschlossene Bestellungen

Zur Entscheidungsfindung wird untersucht, wie viele der betroffenen Bestellungen bereits abgeschlossen wurden.

In [29]:
orders_umsätse_merged = orders_cl.merge(umsätze_zwei_punkte,left_on='order_id', right_on='id_order',how='inner')
orders_umsätse_merged
orders_umsätse_merged['state'].value_counts()

,count
state,
Shopping Basket,24425
Place Order,5245
Completed,3561
Cancelled,1802
Pending,1105


### 3- Preisformat korrigieren

Die erste fehlerhafte Trennstelle wird entfernt, sodass die Preise anschließend als numerische Werte gespeichert werden können.

In [30]:
orderlines_cl.loc[unit_price_mask, 'unit_price'] = orderlines_cl.loc[unit_price_mask, 'unit_price'].str.replace('.','',1)


### 4- Datentyp umwandeln

Nach der Korrektur wird die Spalte in einen numerischen Datentyp umgewandelt.

In [31]:
orderlines_cl['unit_price'] = pd.to_numeric(orderlines_cl['unit_price'])

### 5- Preise an Euro-Format anpassen

Zum Schluss werden die betroffenen Preise auf zwei Dezimalstellen angepasst.

In [32]:
unit_price_mask = orderlines_cl.unit_price.astype(str).str.contains(r"\d+\.\d{3}")
orderlines_cl.loc[unit_price_mask ,'unit_price'] =round(orderlines_cl.loc[unit_price_mask , 'unit_price'], 2)

In [33]:
orderlines_cl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                293983 non-null  int64         
 1   id_order          293983 non-null  int64         
 2   product_id        293983 non-null  int64         
 3   product_quantity  293983 non-null  int64         
 4   sku               293983 non-null  object        
 5   unit_price        293983 non-null  float64       
 6   date              293983 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(4), object(1)
memory usage: 15.7+ MB


## 6. Fehlende Werte prüfen

### 6.1 Bestellungen (`orders`) — identifizieren

In [34]:
orders_cl.isna().sum()

,0
order_id,0
created_date,0
total_paid,5
state,0


**Ergebnis:** `total_paid` hat 5 fehlende Werte — Entscheidung/Behandlung erfolgt in Meilenstein 4, falls relevant für die Umsatzanalyse.

In [35]:
orders_cl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226909 non-null  int64         
 1   created_date  226909 non-null  datetime64[ns]
 2   total_paid    226904 non-null  float64       
 3   state         226909 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 6.9+ MB


In [36]:
orders_cl = orders_cl.dropna(subset='total_paid')
orders_cl.shape[0]

226904

### 6.2 Produkte (`products`) — `price` / `desc`

In [37]:
print("Beispiel-Zeilen mit fehlendem Preis:")
products_cl[products_cl['price'].isna()].head(5)

Beispiel-Zeilen mit fehlendem Preis:


,sku,name,desc,price,promo_price,in_stock,type
34,TWS0019,Twelve South MagicWand support Apple Magic Tra...,MagicWand for wireless keyboard and Magic Trac...,NaN,299.899,0,8696
1900,AII0008,Aiino Case MacBook Air 11 '' Transparent,MacBook Air 11-inch casing with matte finish.,NaN,22.99,0,13835403
2039,CEL0020,Celly Ambo Luxury Leather Case + iPhone 6 Case...,Cover and housing together with magnet for iPh...,NaN,399.905,0,11865403
2042,CEL0007,Celly Wallet Case with removable cover Black i...,Case Book for iPhone 6 card case type.,NaN,128.998,0,11865403
2043,CEL0012,Celly Silicone Hard Shell iPhone 6 Blue,Hard Shell Silicone iPhone 6.,NaN,4.99,0,11865403


In [38]:
products_cl.isna().sum()

,0
sku,0
name,0
desc,7
price,46
promo_price,0
in_stock,0
type,50


**Ergebnis:** Weniger als 0,5 % der Zeilen betroffen → löschen.

In [39]:
percent_corrupted = (100 * (products_cl["price"].isna().sum()) / products_cl.shape[0])
print(f"{percent_corrupted:.2f}% der Zeilen haben einen fehlenden price")

0.43% der Zeilen haben einen fehlenden price


In [40]:
products_cl = products_cl.dropna(subset='price')
products_cl.shape[0]

10534

In [41]:
products_cl.loc[products_cl['desc'].isna(), :]

,sku,name,desc,price,promo_price,in_stock,type
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard ...",NaN,107,814.659,0,1298
16128,APP1622-A,Open - Apple Smart Keyboard Pro Keyboard Folio...,NaN,1.568.206,1.568.206,0,1298
17843,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN,566.35,5.659.896,0,12175397
18152,KAN0034-A,Open - Kanex USB-C Gigabit Ethernet Adapter Ma...,NaN,29.99,237.925,0,1298
18490,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror an...,NaN,24.99,22.99,1,1515
18612,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN,79.99,56.99,1,1515
18690,HOW0001-A,Open - Honeywell thermostat Lyric zonificador ...,NaN,199.99,1.441.174,0,11905404


In [42]:
products_cl.loc[products_cl['desc'].isna(), 'desc'] = products_cl.loc[products_cl['desc'].isna(), 'name']
products_cl['desc'].isna().sum()

np.int64(0)

**Ergebnis:** `desc` wird bewusst nicht vollständig entfernt/ignoriert — wird später zur Kategorisierung genutzt (siehe Analyse-Notebook).

### 6.3 Bestellpositionen (`orderlines`) — restliche fehlende Werte

In [43]:
orderlines_cl = orderlines_cl.dropna(axis=0)

## 7. Speichern (Meilenstein 3 Abschluss)
Export als `orders_cl.csv`, `orderlines_cl.csv`, `products_cl.csv` → Ordner `2.Data/1.Data_cleaned_01/final_CSV_cleaned/`.

In [45]:
#Bereinigte Daten speichern in Drive

from google.colab import files

# orders_cl.to_csv("orders_cl.csv", index=False)
# files.download("orders_cl.csv")

# orderlines_cl.to_csv("orderlines_cl.csv", index=False)
# files.download("orderlines_cl.csv")

# products_cl.to_csv("products_cl.csv", index=False)
# files.download("products_cl.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>